<a href="https://colab.research.google.com/github/mragsp/Seminar3/blob/main/XAI_hands_on.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explainable AI (XAI) — Hands-on Practice

**Approximate duration:** ~1 hour

Welcome! This notebook puts into practice the concepts from the lecture on *Interpretability and Explainability*. You will:

1. Train an **interpretable** model and a **black-box** model on the same task.
2. Apply **LIME** to explain individual predictions of the black-box model.
3. Apply **SHAP** to compute Shapley values and produce feature importance / summary plots.
4. Compare LIME and SHAP, and reflect on the **current limitations** of explainability techniques.

Sections marked **🧑‍💻 YOUR TURN** contain exercises where you are expected to write code. Everything else is guided. Two *bonus* exercises at the end are harder — try them if you finish early.

> **Tip.** Run the notebook top-to-bottom in order. Some cells depend on variables defined earlier.


## 0. Setup

Install the libraries we need. The first cell is the *only* place you should need to install anything.

- `scikit-learn` — models + dataset loading
- `shap` — Shapley value explanations
- `lime` — LIME explanations
- `pandas`, `numpy`, `matplotlib` — data handling + plotting

If you are running on Google Colab or a fresh Jupyter environment, uncomment the `!pip install` line.


In [ ]:
# Run this cell once. If the packages are already installed you can skip it.
# !pip install scikit-learn>=1.3 shap>=0.46 lime>=0.2 pandas numpy matplotlib --quiet

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import shap
import lime
import lime.lime_tabular

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("shap version :", shap.__version__)
print("sklearn      : imported OK")
print("lime         : imported OK")

## 1. Interpretability vs. Explainability  (≈2 min)

From the lecture:

| Concept | Definition |
|---|---|
| **Interpretable model** | A model whose input→output relationship is *simple enough to be memorised*. Few features, intuitive features, simple relationship. Example: a small linear regression on meaningful variables. |
| **Explainable model** | A non-interpretable (black-box) model to which we **apply techniques** (e.g. LIME, SHAP) to explain its predictions. |

A large linear regression with 1,000 one-hot columns is *not* interpretable, even though it is linear — **few variables + intuitive variables** are essential.

In this notebook we will train both kinds of model on the same data and see the difference in practice.


## 2. The dataset: Adult (Census Income)  (≈5 min)

### Why this dataset?

We use the classic **UCI Adult / Census Income** dataset:

- **Task:** binary classification — does a person earn **more than 50 K USD/year** from 1994 US census data?
- **Size:** ~48 000 rows × 14 features — large enough to make SHAP/LIME *interesting* but small enough to fit on any laptop and for SHAP to finish in seconds.
- **Mix of numeric and categorical features** — perfect for XAI, because explanations become meaningful (e.g. *"being married adds +0.3 to the log-odds"*).
- **Socially relevant** — income prediction is exactly the kind of use case where the presentation's final warning applies: *"There is a false sense of trade-off between performance and interpretability"*.

We subsample to **8 000 rows** so SHAP and LIME complete in under a minute.


In [ ]:
# Download the Adult dataset from OpenML (publicly available, no authentication).
# The first call may take ~30 seconds; subsequent calls are cached by sklearn.
data = fetch_openml("adult", version=2, as_frame=True, parser="auto")

df = data.frame.copy()
print("Full shape :", df.shape)
print("Columns    :", list(df.columns))
df.head()

In [ ]:
# Subsample for speed (keeps class balance)
df = df.sample(n=8000, random_state=RANDOM_STATE).reset_index(drop=True)

# Inspect
print("Shape after subsampling :", df.shape)
print("\nTarget distribution:")
print(df["class"].value_counts(normalize=True).round(3))
print("\nDtypes:")
print(df.dtypes)

### 🧑‍💻 YOUR TURN — Quick data check  (1 min)

Before training anything, do a sanity check:

1. How many **missing values** are there per column?
2. What is the **mean age** of people earning `>50K` vs `<=50K`?

Write the two lines of pandas below. This is just a warm-up so you are comfortable with the dataframe `df`.


In [ ]:
# YOUR CODE HERE ------------------------------------------------------------
# 1) Missing values per column:
# print(df.isna().sum())

# 2) Mean age by income class:
# print(df.groupby("class")["age"].mean())
# ---------------------------------------------------------------------------

## 3. Preprocessing  (≈3 min, guided)

- Fill missing values in categorical columns with the string `"Unknown"`.
- Drop `fnlwgt` (a sampling weight — not predictive).
- **Label-encode** each categorical column (tree models don't need one-hot encoding, and label encoding keeps column count small, which makes SHAP/LIME plots much more readable).
- Encode target as 0/1.

We keep a `category_maps` dictionary so that, later, we can translate the numeric codes back to the original string labels when interpreting explanations — important for LIME and SHAP readability.


In [ ]:
# Drop non-informative column
df = df.drop(columns=["fnlwgt"])

# Identify column types
target_col = "class"
categorical_cols = df.select_dtypes(include=["category", "object"]).columns.tolist()
categorical_cols.remove(target_col)
numeric_cols = [c for c in df.columns if c not in categorical_cols + [target_col]]

print("Categorical features:", categorical_cols)
print("Numeric features    :", numeric_cols)

In [ ]:
# Fill missing values and label-encode categorical features
category_maps = {}  # col -> LabelEncoder, needed to decode later

for col in categorical_cols:
    df[col] = df[col].astype("object").fillna("Unknown")
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    category_maps[col] = le

# Target encoding: ">50K" -> 1, "<=50K" -> 0
df[target_col] = (df[target_col].astype(str).str.strip() == ">50K").astype(int)

X = df.drop(columns=[target_col])
y = df[target_col].values
feature_names = X.columns.tolist()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("Positive class (>50K) proportion in train:", y_train.mean().round(3))

## 4. An interpretable model vs. a black-box model  (≈8 min, guided)

From the lecture: *"Not all linear regressions are interpretable"* — but a **small logistic regression on meaningful features** certainly is. We will also train a **Random Forest** as the black-box counterpart.


In [ ]:
# --- Interpretable model: Logistic Regression on numeric features only ---
# (Small number of features + intuitive + simple input->output mapping.)
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train[numeric_cols], y_train)
lr_acc = accuracy_score(y_test, lr.predict(X_test[numeric_cols]))

# --- Black-box model: Random Forest on ALL features ---
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_acc = accuracy_score(y_test, rf.predict(X_test))

print(f"Logistic Regression accuracy : {lr_acc:.3f}")
print(f"Random Forest accuracy       : {rf_acc:.3f}")

In [ ]:
# The Logistic Regression IS its own explanation: each coefficient is the
# contribution of the corresponding standardised feature to the log-odds.
lr_coef = pd.Series(lr.coef_[0], index=numeric_cols).sort_values()

fig, ax = plt.subplots(figsize=(7, 3))
lr_coef.plot.barh(ax=ax, color="#1f77b4")
ax.set_title("Logistic Regression coefficients (numeric features)")
ax.axvline(0, color="black", lw=0.8)
plt.tight_layout()
plt.show()

**Expected output.**
- Logistic Regression accuracy ≈ **0.80**.
- Random Forest accuracy ≈ **0.85**.
- The Random Forest wins by several points, but it is a *black box* — we cannot read its 200 trees to understand *why* a given person is predicted to earn >50K.
- In the LR coefficient plot, features like `education-num`, `capital-gain`, `hours-per-week` should come out positive; `capital-loss` sign depends on encoding.

This is the setting the lecture describes: we gain predictive power but lose direct interpretability, and we therefore need **XAI techniques**.


## 5. LIME — Local Interpretable Model-agnostic Explanations  (≈12 min)

**Reminder from the lecture.** For each prediction, LIME:

1. Samples synthetic points **around the input**.
2. Runs them through the black-box model to get predictions.
3. Fits a **weighted linear regression** to those points (closer points weigh more).
4. Reports the linear coefficients as the local explanation.

The explanation is therefore **local** (valid only near the instance) and **model-agnostic** (only needs `predict_proba`).


In [ ]:
# Build the LIME explainer once. It needs training data to know the
# distribution of each feature (for perturbation) and the indices of
# categorical columns so that it doesn't produce nonsensical perturbed values.
categorical_indices = [X_train.columns.get_loc(c) for c in categorical_cols]

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_names,
    class_names=["<=50K", ">50K"],
    categorical_features=categorical_indices,
    mode="classification",
    random_state=RANDOM_STATE,
)
print("LIME explainer ready.")

In [ ]:
# Pick one instance from the test set and ask LIME to explain it.
idx = 0
instance = X_test.iloc[idx].values
true_label = ["<=50K", ">50K"][y_test[idx]]
pred_proba = rf.predict_proba(instance.reshape(1, -1))[0]
predicted_label = ["<=50K", ">50K"][int(pred_proba.argmax())]

print(f"Instance #{idx}")
print(f"  true label      : {true_label}")
print(f"  predicted label : {predicted_label}")
print(f"  P(>50K)         : {pred_proba[1]:.3f}")

lime_exp = lime_explainer.explain_instance(
    instance, rf.predict_proba, num_features=8, num_samples=5000
)

print("\nTop features pushing the prediction (LIME):")
for feature, weight in lime_exp.as_list():
    direction = "↑ >50K" if weight > 0 else "↓ <=50K"
    print(f"  {direction}  {weight:+.3f}   {feature}")

In [ ]:
# Matplotlib version of the LIME explanation (readable in any environment)
fig = lime_exp.as_pyplot_figure()
fig.set_size_inches(8, 4)
plt.title(f"LIME — instance #{idx}  (P(>50K)={pred_proba[1]:.2f})")
plt.tight_layout()
plt.show()

**How to read the plot.**
- Each bar is a *condition* on one feature (LIME discretises numeric features into quartile bins).
- **Green / positive** bars push the prediction towards `>50K`.
- **Red / negative** bars push it towards `<=50K`.
- The sum of bars + the local model's intercept ≈ the black-box probability.

For this particular row you should see the strongest signals on `capital-gain`, `marital-status`, `education-num`, `age` or `hours-per-week`, with signs that agree with common sense.


### 🧑‍💻 YOUR TURN — Explain a *misclassified* prediction  (≈5 min)

LIME is most useful when we want to understand **mistakes**. Write code that:

1. Finds one test-set row where `rf.predict(...)` disagrees with `y_test`.
2. Uses `lime_explainer.explain_instance(...)` with `num_features=6` to explain it.
3. Prints the explanation as a list (`.as_list()`).

*Hint.* `np.where(rf.predict(X_test) != y_test)[0]` gives the indices of all wrongly-predicted rows — pick the first one.


In [ ]:
# YOUR CODE HERE ------------------------------------------------------------
# Step 1: find a misclassified index
# wrong_idx = ...

# Step 2: get the instance and explain it
# wrong_instance = X_test.iloc[wrong_idx].values
# wrong_exp = lime_explainer.explain_instance(wrong_instance, rf.predict_proba, num_features=6, num_samples=5000)

# Step 3: print the explanation
# print("True label:", y_test[wrong_idx], " Predicted:", rf.predict(wrong_instance.reshape(1, -1))[0])
# for f, w in wrong_exp.as_list():
#     print(f"  {w:+.3f}  {f}")
# ---------------------------------------------------------------------------

**What to look for.** When a model misclassifies, LIME often highlights a feature whose value is *atypical* for the predicted class. E.g. the model predicts `<=50K` but `capital-gain` is large — LIME will flag it as the feature that *should* have pushed the other way. This is exactly the kind of local error analysis XAI is built for.


## 6. SHAP — Shapley values  (≈10 min, guided)

### The three axioms

The lecture introduced SHAP via three properties that **uniquely determine** the Shapley values:

1. **Local accuracy** — for any single prediction, the sum of the feature attributions plus the expected model output equals the actual model output.
   $$f(x) = \phi_0 + \sum_{i} \phi_i$$
2. **Missingness** — a feature that is *constantly missing* (i.e. has no effect) gets attribution 0.
3. **Consistency** — if we change the model so that a feature's marginal contribution increases (or stays the same) in every coalition, that feature's SHAP value will not decrease.

For tree ensembles, `TreeExplainer` computes exact Shapley values in polynomial time.


In [ ]:
# TreeExplainer is fast because it exploits the tree structure. For a
# RandomForest of 200 trees and 13 features, ~500 rows takes ~3 s on a laptop.
explainer = shap.TreeExplainer(rf)

# Explain a subset of the test set (enough for stable global plots)
X_explain = X_test.iloc[:500]
shap_values = explainer(X_explain)

# Binary classifier -> SHAP returns shape (N, F, 2).  We slice to get the
# positive class ">50K".
shap_values_pos = shap_values[:, :, 1]

print("shap_values.values shape :", shap_values.values.shape)
print("shap_values_pos.values   :", shap_values_pos.values.shape)
print("expected value (baseline):", shap_values_pos.base_values[0].round(3))

### Local accuracy in practice

Let's **verify axiom #1 numerically** on one instance: the sum of SHAP values plus the base value should equal the model's prediction (raw probability for the positive class).


In [ ]:
i = 0
phi = shap_values_pos.values[i]        # SHAP values for this row, positive class
phi0 = shap_values_pos.base_values[i]  # baseline (expected model output)
pred = rf.predict_proba(X_explain.iloc[[i]])[0, 1]
sum_shap = phi0 + phi.sum()

print(f"Baseline (E[f(x)])        : {phi0:.4f}")
print(f"Sum of SHAP values        : {phi.sum():+.4f}")
print(f"Baseline + sum of SHAP    : {sum_shap:.4f}")
print(f"Model prediction P(>50K)  : {pred:.4f}")
print(f"Match within 1e-6 ?       : {np.isclose(sum_shap, pred, atol=1e-6)}")

In [ ]:
# Waterfall plot for the same row — the most intuitive local view.
shap.plots.waterfall(shap_values_pos[0], max_display=10, show=True)

**Expected output.**
- `Baseline + sum of SHAP == Model prediction` should hold to ~6 decimals — that is *local accuracy* (axiom 1) with your own eyes.
- The waterfall plot shows each feature pushing the prediction up (red) or down (blue) from the baseline `E[f(x)]` to the final `f(x)`.


## 7. SHAP feature importance  (≈6 min)

### Concept

The **global** feature importance from SHAP is just the **mean of the absolute SHAP values** over all explained instances, per feature:
$$\text{importance}_j = \frac{1}{N}\sum_{i=1}^{N} |\phi_j^{(i)}|$$

This is the quantity displayed in `shap.plots.bar(...)`.


### 🧑‍💻 YOUR TURN — Compute it yourself  (≈4 min)

Without using `shap.plots.bar`, build the same ranking manually:

1. Take `shap_values_pos.values` (shape `(N, F)`).
2. Compute the mean absolute value along axis 0 — one number per feature.
3. Put them into a `pandas.Series` indexed by `feature_names`, sort descending.
4. Plot the top 10 as a horizontal bar chart.

Then call `shap.plots.bar(shap_values_pos, max_display=10)` and check that the order of features matches.


In [ ]:
# YOUR CODE HERE ------------------------------------------------------------
# Step 1 & 2: mean absolute SHAP per feature
# mean_abs_shap = np.abs(shap_values_pos.values).mean(axis=0)

# Step 3: as a sorted pandas Series
# importance = pd.Series(mean_abs_shap, index=feature_names).sort_values(ascending=True)

# Step 4: plot the top 10
# fig, ax = plt.subplots(figsize=(7, 4))
# importance.tail(10).plot.barh(ax=ax, color="#d62728")
# ax.set_title("SHAP feature importance (mean |SHAP|)")
# ax.set_xlabel("mean(|SHAP value|)")
# plt.tight_layout()
# plt.show()

# Step 5: compare with the built-in
# shap.plots.bar(shap_values_pos, max_display=10)
# ---------------------------------------------------------------------------

**Expected output.** On Adult + Random Forest, the most important features are usually (order may vary slightly): `marital-status`, `capital-gain`, `education-num`, `age`, `relationship`, `hours-per-week`. Your bar chart and `shap.plots.bar` should show the same top features in the same order.


## 8. SHAP summary plot (beeswarm)  (≈6 min)

The summary plot shows **every SHAP value for every instance**: each row is a feature, each dot is a person in the test set. The *x-position* is the SHAP value (impact on the prediction), the *colour* is the feature value (red = high, blue = low).

This single plot encodes **three pieces of information** per feature:
- global importance (spread left/right),
- direction of effect (positive or negative SHAP),
- whether high or low feature values push the prediction up.


In [ ]:
shap.plots.beeswarm(shap_values_pos, max_display=10, show=True)

### 🧑‍💻 YOUR TURN — Read the plot  (≈3 min)

Look at the `age` row in the beeswarm plot and answer **in a short comment in the cell below**:

1. Are most high-age points (red) on the positive or negative side of 0?
2. What does that tell you about the effect of `age` on the probability of earning `>50K` according to this Random Forest?
3. Does the same pattern hold for `education-num`?


In [ ]:
# YOUR ANSWER (write as comments) -------------------------------------------
# 1)
# 2)
# 3)
# ---------------------------------------------------------------------------

**Expected answer.** Red points (high age) sit on the right-hand, positive side ⇒ being older *increases* the predicted probability of `>50K`. Same pattern for `education-num`: more years of education push the prediction up. The beeswarm makes these relationships visible in a single glance, which bar plots do not.


## 9. LIME vs SHAP on the same instance  (≈3 min, guided)

Let's explain the **same row** with both methods and compare.


In [ ]:
# Same test row for both
i = 7
row = X_test.iloc[i]
instance = row.values

# LIME explanation
lime_local = lime_explainer.explain_instance(
    instance, rf.predict_proba, num_features=8, num_samples=5000
)
lime_pairs = lime_local.as_list()

# SHAP explanation (positive class)
shap_vals_i = explainer(row.to_frame().T)[:, :, 1].values[0]
shap_pairs = sorted(
    zip(feature_names, shap_vals_i), key=lambda t: abs(t[1]), reverse=True
)[:8]

print(f"P(>50K) = {rf.predict_proba(instance.reshape(1,-1))[0,1]:.3f}\n")

print(f"{'LIME (weight, condition)':45s}   SHAP (value, feature)")
print("-" * 90)
for (lf, lw), (sf, sv) in zip(lime_pairs, shap_pairs):
    print(f"{lw:+7.3f}  {lf:<35s}   {sv:+7.3f}  {sf}")

**What to notice.**

- Both methods typically agree on the **top 2–3 features**.
- The **values differ**: LIME gives the *local linear coefficient*, SHAP gives the *exact Shapley value* — they have different mathematical meanings.
- LIME groups numeric features into *bins* (e.g. `"37 < age ≤ 47"`), whereas SHAP keeps the raw value.
- LIME is **stochastic** (runs twice → slightly different results), SHAP (via `TreeExplainer`) is **deterministic** for tree models.


## 10. Current limitations of explainability  (discussion)

From the last two slides of the lecture:

1. *"There is a false sense of trade-off between performance and interpretability."*
    - In our own results, LR got ~80% and RF got ~85% — a real but modest gap. On many problems, a well-regularised linear model or a small decision tree is within 1–2 points of a black box. Reaching for XAI *just because* a black box wins by 2% is often a bad trade.
2. *"Not all explanations are satisfactory."*
    - Explanations depend on the **reference / background distribution** (both SHAP and LIME) — change it and the explanation changes.
    - LIME is unstable: the next exercise shows this.
    - SHAP values are *correct given the feature set*, but features are often **correlated**, so the attribution between two correlated features is ambiguous.
    - An explanation tells you *what* the model did, not *whether it was right to do so* — XAI is not a substitute for validation, fairness auditing, or causal analysis.


---
## 🎁 Bonus 1 — LIME stability  (≈7 min, harder)

LIME generates perturbations **randomly**, so two consecutive runs on the same instance produce different coefficients. How different?

**Your task.**
1. Pick one test instance (e.g. `idx = 0`).
2. Call `lime_explainer.explain_instance(..., num_features=8, num_samples=1000)` **10 times** — *do not* set the random seed between runs (we *want* the variability). You can do this by creating a new explainer each time with a different `random_state`, or by passing `random_state=None` and reusing the same explainer.
3. For each run, extract the weight of the feature named `"age"` (it may appear inside a condition like `"37 < age <= 47"` — use `"age" in feature_desc`).
4. Report the **mean** and **standard deviation** of the 10 weights.
5. Repeat with `num_samples=10_000` and compare — does stability improve?

This exercise reproduces a well-known criticism of LIME: explanations can vary noticeably between runs.


In [ ]:
# YOUR CODE HERE ------------------------------------------------------------
# idx = 0
# instance = X_test.iloc[idx].values

# def age_weight(num_samples, n_runs=10):
#     weights = []
#     for seed in range(n_runs):
#         explainer_s = lime.lime_tabular.LimeTabularExplainer(
#             X_train.values, feature_names=feature_names,
#             class_names=["<=50K", ">50K"],
#             categorical_features=categorical_indices,
#             mode="classification", random_state=seed,
#         )
#         exp_s = explainer_s.explain_instance(
#             instance, rf.predict_proba, num_features=8, num_samples=num_samples
#         )
#         for desc, w in exp_s.as_list():
#             if "age" in desc:
#                 weights.append(w)
#                 break
#     return np.array(weights)
#
# w_small = age_weight(1000)
# w_large = age_weight(10000)
# print(f"num_samples=1000   mean={w_small.mean():+.3f}  std={w_small.std():.3f}")
# print(f"num_samples=10000  mean={w_large.mean():+.3f}  std={w_large.std():.3f}")
# ---------------------------------------------------------------------------

**Expected outcome.** The standard deviation with `num_samples=1000` is noticeably larger than with `num_samples=10_000` — typically 2–5× smaller std with more samples. The *sign* is usually stable; the *magnitude* is the part that fluctuates. This is a direct illustration of the lecture's point that "not all explanations are satisfactory".


---
## 🎁 Bonus 2 — Compute Shapley values by hand  (≈10 min, harder)

The Shapley value of feature $i$ for prediction $f(x)$ is:

$$\phi_i(f, x) = \sum_{S \subseteq N \setminus \{i\}} \frac{|S|!\,(|N|-|S|-1)!}{|N|!}\bigl[\, f_{S\cup\{i\}}(x) - f_S(x) \,\bigr]$$

where $f_S$ is the model's expected output when only the features in $S$ are known (and the others are marginalised over a background distribution).

For **3 features**, the sum over $S$ has only $2^{3-1} = 4$ terms per feature, so we can implement it by brute force.

**Your task.**
1. Use `X_train[["age", "education-num", "hours-per-week"]]` only.
2. Train a new small `RandomForestClassifier(n_estimators=100, max_depth=5)` on those 3 features.
3. Use **100 background samples** from `X_train[["age", "education-num", "hours-per-week"]]` as the reference distribution.
4. For **one** test instance, compute $\phi_i$ for each of the 3 features using the brute-force formula above.
5. Verify against `shap.TreeExplainer(small_rf, background).shap_values(...)` — the two should match up to a small numerical error.

This exercise makes the *definition* of Shapley values concrete. A helper skeleton is provided.


In [ ]:
# YOUR CODE HERE ------------------------------------------------------------
from itertools import combinations
from math import factorial

# 1) Subset the data
# cols3 = ["age", "education-num", "hours-per-week"]
# Xtr3 = X_train[cols3].values
# Xte3 = X_test[cols3].values

# 2) Train a small RF on those 3 features
# small_rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=RANDOM_STATE)
# small_rf.fit(Xtr3, y_train)

# 3) Background = 100 random training rows
# rng = np.random.default_rng(0)
# background = Xtr3[rng.choice(len(Xtr3), 100, replace=False)]

# 4) Brute-force Shapley value for feature i on instance x
# def fS(S, x, background, model):
#     # expected model output with features in S fixed, others ~ background
#     data = np.tile(background, (1, 1)).astype(float)
#     for j in S:
#         data[:, j] = x[j]          # fix feature j at x's value
#     return model.predict_proba(data)[:, 1].mean()
#
# def shapley_feature(i, x, background, model, n_features=3):
#     others = [j for j in range(n_features) if j != i]
#     total = 0.0
#     for k in range(len(others) + 1):
#         for S in combinations(others, k):
#             S = list(S)
#             marginal = fS(S + [i], x, background, model) - fS(S, x, background, model)
#             weight = factorial(k) * factorial(n_features - k - 1) / factorial(n_features)
#             total += weight * marginal
#     return total
#
# x0 = Xte3[0]
# my_phi = [shapley_feature(i, x0, background, small_rf) for i in range(3)]
# print("Brute-force Shapley values:", np.round(my_phi, 4))

# 5) Compare with SHAP
# shap_small = shap.TreeExplainer(small_rf, background, feature_perturbation="interventional")
# shap_vals_small = shap_small.shap_values(x0.reshape(1, -1))
# If binary, shap_vals_small may be either shape (1, 3) or (1, 3, 2) depending on version;
# pick the positive class in the second case.
# print("SHAP library values      :", np.round(shap_vals_small, 4))
# ---------------------------------------------------------------------------

**Expected outcome.** Your brute-force Shapley values for the 3 features should agree with `shap.TreeExplainer(..., feature_perturbation="interventional")` up to ~1e-3 (the only source of error is the 100-sample Monte-Carlo background).

**What you just proved.** SHAP is not magic — it is a well-defined average of marginal contributions over all feature subsets. For 3 features this fits in 4 lines of code. `TreeExplainer` just computes the same thing without having to enumerate $2^F$ coalitions, using tree structure to factorise the sum.


---
## ✅ You're done.

**Recap — what you practiced**
- Trained an interpretable model (LR on numeric features) and a black box (RF) to see the *true* performance gap.
- Used **LIME** to explain both correct and incorrect individual predictions, and observed its stochasticity.
- Used **SHAP** to compute exact Shapley values on a tree ensemble, verified **local accuracy** numerically, and built your own feature-importance bar chart.
- Read a SHAP **beeswarm / summary plot** to recover *direction* and *strength* of each feature's effect.
- Compared LIME and SHAP on the same instance and reflected on their current limitations.

If you want to push further: try replacing the Random Forest with `sklearn.ensemble.GradientBoostingClassifier` and re-running the SHAP section — does the feature ranking stay the same? Should it?
